# 道路修復資料前處理

* 資料說明 : 蒐集公路總局養護工程處 2014~2026 合計 13 年的資料

* 資料來源 : 政府資料開放平台 https://data.gov.tw/dataset/31020

# 1. 資料讀取

In [1]:
import pandas as pd
import os

DATA_DIR = "raw_data"

# 載入原始資料
df_raw = pd.read_csv(os.path.join(DATA_DIR, "road_raw.csv"))

# 欄位與資料筆數
print(f"資料筆數：{len(df_raw)}")
print(f"欄位數：{len(df_raw.columns)}")

# Check
df_raw.head()

資料筆數：16014
欄位數：12


,轄管局處段,通報時間,管制時間,實際解除時間,事件名稱,路線,災害通報類別,災害主次類別,管制原因,管制作為,替代路線圖,災情照片
0,交通部公路局 北區養護工程分局 復興工務段,2014-06-05 06:52:00,NaN,2014-06-05 09:00:00,103年06月災情,省道台7線37K+700~37K+700,NaN,道路 道路落石,零星落石,NaN,NaN,\n
1,交通部公路局 北區養護工程分局 復興工務段,2014-06-06 20:40:00,NaN,2014-06-06 22:00:00,103年06月災情,省道台7線35K+500~35K+500,NaN,道路 道路落石,零星落石,NaN,NaN,\n
2,交通部公路局 北區養護工程分局 復興工務段,2014-06-23 07:14:00,NaN,2014-06-23 09:55:00,103年06月災情,省道台7線19K+500~19K+500,NaN,道路 道路落石,零星落石,NaN,NaN,\n
3,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2014-06-01 01:00:00,NaN,2014-06-01 03:00:00,103年06月災情,省道台18線72K+500~72K+500,NaN,道路 道路落石,零星落石,NaN,NaN,\n
4,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2014-06-09 07:20:00,NaN,2014-06-09 09:20:00,103年06月災情,省道台18線72K+500~72K+500,NaN,道路 道路落石,零星落石,NaN,NaN,\n


# 2. Data Cleaning

## 2.1 Deal with duplicated data

In [2]:
# 重複資料檢查
dup_count = df_raw.duplicated().sum()
print(f"完全重複筆數：{dup_count}")

# 移除重複資料，保留第一筆
df_raw = df_raw.drop_duplicates(keep="first").reset_index(drop=True)
print(f"移除重複後筆數：{len(df_raw)}")

完全重複筆數：175
移除重複後筆數：15839


## 2.2 Missing Data Check

* 主要關注於通報時間與實際解除時間，這會影響到路段修復時間的估計

In [3]:
# 缺值檢核
null_info = pd.DataFrame({
    "null_count" : df_raw.isna().sum(),
    "null_pct"   : (df_raw.isna().sum() / len(df_raw) * 100).round(1),
})
null_info

,null_count,null_pct
轄管局處段,1,0.0
通報時間,0,0.0
管制時間,7247,45.8
實際解除時間,491,3.1
事件名稱,0,0.0
路線,0,0.0
災害通報類別,7244,45.7
災害主次類別,0,0.0
管制原因,15,0.1
管制作為,7866,49.7


# 3. Feature Selection

| 欄位         | 保留 | 原因                        |
|--------------|------|-----------------------------|
| 轄管局處段   | ✅   |   |
| 通報時間     | ✅   | Duration 起始時間           |
| 實際解除時間 | ✅   | Duration 結束時間           |
| 路線         | ✅   |   |
| 災害主次類別 | ✅   | Cause 來源欄位              |
| 管制原因     | ✅   | 災害描述，供人工核查        |
| 管制時間     | ❌   | 改以通報時間為起點，不再使用 |
| 事件名稱     | ❌   | 都是寫幾月份事件沒必要 |
| 災害通報類別 | ❌   | 缺值 46%，以災害主次類別取代 |
| 管制作為     | ❌   | 缺值過多且非分析欄位        |
| 替代路線圖   | ❌   | 100% 缺值，原始資料為 KML 檔案才具備照片 |
| 災情照片     | ❌   | 分析無用，原始資料為 KML 檔案才具備照片  |

In [4]:
cols_keep = [
    "轄管局處段",
    "通報時間",
    "實際解除時間",
    "路線",
    "災害主次類別",
    "管制原因",
]

df = df_raw[cols_keep].copy()

print(f"欄位數：{len(df.columns)}")
print(f"筆數：{len(df)}")

# Check
df.head()

欄位數：6
筆數：15839


,轄管局處段,通報時間,實際解除時間,路線,災害主次類別,管制原因
0,交通部公路局 北區養護工程分局 復興工務段,2014-06-05 06:52:00,2014-06-05 09:00:00,省道台7線37K+700~37K+700,道路 道路落石,零星落石
1,交通部公路局 北區養護工程分局 復興工務段,2014-06-06 20:40:00,2014-06-06 22:00:00,省道台7線35K+500~35K+500,道路 道路落石,零星落石
2,交通部公路局 北區養護工程分局 復興工務段,2014-06-23 07:14:00,2014-06-23 09:55:00,省道台7線19K+500~19K+500,道路 道路落石,零星落石
3,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2014-06-01 01:00:00,2014-06-01 03:00:00,省道台18線72K+500~72K+500,道路 道路落石,零星落石
4,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2014-06-09 07:20:00,2014-06-09 09:20:00,省道台18線72K+500~72K+500,道路 道路落石,零星落石


# 4. 災害類別確認

* 確認災害主次類別是否皆為「意外」造成
 
* 另外由於橋梁與隧道的工程數據較少，暫不額外進行分析

In [5]:
# 災害主次類別 分布統計
cause_dist = (
    df["災害主次類別"]
    .value_counts(dropna=False)
    .rename_axis("災害主次類別")
    .reset_index(name="count")
)
cause_dist["pct"] = (cause_dist["count"] / len(df) * 100).round(1)
cause_dist

,災害主次類別,count,pct
0,道路 道路落石,9720,61.4
1,道路 預警性封閉,3186,20.1
2,道路 邊坡坍方,1052,6.6
3,道路 其他,1052,6.6
4,道路 交通事故(含危險品洩漏疑慮、火災),197,1.2
5,道路 淹水,180,1.1
6,道路 土石流阻斷,136,0.9
7,道路 路基流失,88,0.6
8,其他 其他,57,0.4
9,橋梁 預警性封閉,47,0.3


# 5. 排除預警性封閉

由於資料主要著重在分析，受損後的修復時間，移除所有含預警性封閉非必要資料

In [6]:
# 排除預警性封閉
mask_excl = df["災害主次類別"].str.contains("預警性封閉", na=False)
print(f"預警性封閉筆數：{mask_excl.sum()}")

df = df[~mask_excl].reset_index(drop=True)
print(f"排除後筆數：{len(df)}")

預警性封閉筆數：3235
排除後筆數：12604


# 6. 移除空值

* 移除 實際解除時間 為空的資料，避免計算修復時間時出現問題

In [7]:
# 移除實際解除時間為空的列
null_count = df["實際解除時間"].isna().sum()
print(f"實際解除時間空值筆數：{null_count}")

df = df.dropna(subset=["實際解除時間"]).reset_index(drop=True)
print(f"移除後筆數：{len(df)}")

實際解除時間空值筆數：390
移除後筆數：12214


# 7. Datetime 轉換

* 以利後續做時間間格計算

In [8]:
# 通報時間、實際解除時間 轉換為 datetime
df["通報時間"]     = pd.to_datetime(df["通報時間"],     errors="coerce")
df["實際解除時間"] = pd.to_datetime(df["實際解除時間"], errors="coerce")

# Check
df

,轄管局處段,通報時間,實際解除時間,路線,災害主次類別,管制原因
0,交通部公路局 北區養護工程分局 復興工務段,2014-06-05 06:52:00,2014-06-05 09:00:00,省道台7線37K+700~37K+700,道路 道路落石,零星落石
1,交通部公路局 北區養護工程分局 復興工務段,2014-06-06 20:40:00,2014-06-06 22:00:00,省道台7線35K+500~35K+500,道路 道路落石,零星落石
2,交通部公路局 北區養護工程分局 復興工務段,2014-06-23 07:14:00,2014-06-23 09:55:00,省道台7線19K+500~19K+500,道路 道路落石,零星落石
3,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2014-06-01 01:00:00,2014-06-01 03:00:00,省道台18線72K+500~72K+500,道路 道路落石,零星落石
4,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2014-06-09 07:20:00,2014-06-09 09:20:00,省道台18線72K+500~72K+500,道路 道路落石,零星落石
...,...,...,...,...,...,...
12209,交通部公路局 南區養護工程分局 甲仙工務段,2026-05-30 06:35:00,2026-05-30 08:40:00,高雄市桃源區省道台20臨105線19K+900~19K+900,道路 其他,因路樹倒塌導致道路阻斷無法通行，經現場人員積極搶修，目前恢復雙向通行
12210,交通部公路局 中區養護工程分局 信義工務段,2026-05-30 12:36:00,2026-05-30 14:25:00,南投縣信義鄉省道台21線88K+0~88K+0,道路 其他,路權外樹木倒塌 占用路面
12211,交通部公路局 南區養護工程分局 潮州工務段,2026-05-30 16:41:00,2026-05-30 18:28:00,屏東縣屏東市省道台1線399K+950~399K+950,道路 其他,路樹倒塌已排除，道路恢復正常通行。
12212,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2026-05-30 06:50:00,2026-05-30 09:57:00,嘉義縣阿里山鄉省道台18線78K+800~78K+800,道路 道路落石,落石、倒樹佔據東行車道，西行車道可通行


# 8. 新增 Duration 欄位

In [9]:
# Duration (minutes) = 實際解除時間 − 通報時間
df["Duration (minutes)"] = (
    df["實際解除時間"] - df["通報時間"]
).dt.total_seconds() / 60

# 確認
print(df["Duration (minutes)"].describe().round(2))
df[["通報時間", "實際解除時間", "Duration (minutes)"]].head()

count      12214.00
mean         360.90
std        24591.25
min     -1484648.45
25%            8.00
50%           65.00
75%          138.00
max      1600817.47
Name: Duration (minutes), dtype: float64


,通報時間,實際解除時間,Duration (minutes)
0,2014-06-05 06:52:00,2014-06-05 09:00:00,128.0
1,2014-06-06 20:40:00,2014-06-06 22:00:00,80.0
2,2014-06-23 07:14:00,2014-06-23 09:55:00,161.0
3,2014-06-01 01:00:00,2014-06-01 03:00:00,120.0
4,2014-06-09 07:20:00,2014-06-09 09:20:00,120.0


# 9. 處理異常值

## 9.1 異常資料檢視

In [10]:
# 負值
print("=== 負值筆數 ===")
print((df["Duration (minutes)"] <= 0).sum())

# 超大值（超過 30 天 = 43,200 分鐘）
print("\n=== 超過 30 天筆數 ===")
print((df["Duration (minutes)"] > 43_200).sum())

# 預覽極端值
df[df["Duration (minutes)"] <= 0 ][["通報時間", "實際解除時間", "Duration (minutes)", "災害主次類別"]]

=== 負值筆數 ===
1921

=== 超過 30 天筆數 ===
20


,通報時間,實際解除時間,Duration (minutes),災害主次類別
6,2014-06-15 00:00:00,2014-06-15 00:00:00,0.000000,道路 道路落石
7,2014-06-17 00:00:00,2014-06-17 00:00:00,0.000000,道路 道路落石
8,2014-06-17 00:00:00,2014-06-17 00:00:00,0.000000,道路 道路落石
9,2014-06-17 00:00:00,2014-06-17 00:00:00,0.000000,道路 道路落石
10,2014-06-17 00:00:00,2014-06-17 00:00:00,0.000000,道路 道路落石
...,...,...,...,...
12070,2026-01-01 16:14:19,2026-01-01 16:09:00,-5.316667,道路 交通事故(含危險品洩漏疑慮、火災)
12073,2026-01-10 12:48:36,2026-01-10 12:30:00,-18.600000,道路 交通事故(含危險品洩漏疑慮、火災)
12087,2026-02-01 10:00:20,2026-02-01 09:58:00,-2.333333,道路 交通事故(含危險品洩漏疑慮、火災)
12194,2026-05-18 16:06:36,2026-05-18 16:04:00,-2.600000,道路 交通事故(含危險品洩漏疑慮、火災)


## 9.2 異常值處理 - 負值

* 移除負值資料 : 應為人工填寫錯誤

In [11]:
# 移除 Duration ≤ 0
neg_count = (df["Duration (minutes)"] <= 0).sum()
print(f"Duration ≤ 0 筆數：{neg_count}")

df = df[df["Duration (minutes)"] > 0].reset_index(drop=True)
print(f"移除後筆數：{len(df)}")

Duration ≤ 0 筆數：1921
移除後筆數：10293


## 9.3 異常值處理 - 修復時間過長者

* 人工檢視超過 30 天修復時間的資料

* 主要問題同樣為人為資料填入錯誤導致(人工填表時選錯 月份或年分)，此處主要的災害類別為 : 路基流失、道路落石、邊坡坍方導致，其中 道路落石 導致的長修復時間需要被逐筆檢視，因為通常 道路落石不應含有這麼長的修復時間

* 此處暫不考慮使用 ATC-13 所訂定的 90 天作為上限閥值，主因為實際資料可以發現，造成道路中斷超過 90 天者，在邊坡坍方及路基流失的事件實屬合理

* 不考慮移除極值或採用平均值替待之，因災害本來就具有極端事件

In [12]:
# 新增天數欄位方便檢核
df["Duration (days)"] = (df["Duration (minutes)"] / 1440).round(1)

# 篩選超過 30 天者做檢視 
df[df["Duration (days)"] > 30].sort_values("災害主次類別")

,轄管局處段,通報時間,實際解除時間,路線,災害主次類別,管制原因,Duration (minutes),Duration (days)
9199,交通部公路局 南區養護工程分局 關山工務段,2024-07-04 08:00:00,2024-08-03 17:00:00,臺東縣海端鄉省道台20線170K+100~170K+100,道路 路基流失,下邊坡路基坍滑，雙向道路正常通行。,4.374000e+04,30.4
6961,交通部公路局 中區養護工程分局 埔里工務段,2023-08-06 10:14:53,2023-09-12 17:00:00,南投縣仁愛鄉省道台14線95K+100~95K+100,道路 路基流失,已搶通，單線雙向通行,5.368512e+04,37.3
6962,交通部公路局 中區養護工程分局 埔里工務段,2023-08-06 10:20:50,2023-09-12 17:00:00,南投縣仁愛鄉省道台14線96K+200~96K+200,道路 路基流失,已搶通，單線雙向通行,5.367917e+04,37.3
7401,交通部公路局 南區養護工程分局 關山工務段,2023-07-30 17:00:00,2023-11-25 07:00:00,臺東縣海端鄉省道台20臨105線38K+800~39K+200,道路 路基流失,關山工務段已於112-11-25 07-00排除,1.693200e+05,117.6
292,交通部公路局 中區養護工程分局 谷關工務段,2017-06-14 09:11:00,2018-07-01 07:00:00,省道台8線109K+000~109K+000,道路 道路落石,NaN,5.499490e+05,381.9
735,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2018-08-29 00:19:00,2018-09-29 15:30:00,省道台18線35K+400~35K+400,道路 道路落石,邊坡坍土,4.555100e+04,31.6
1234,交通部公路局 北區養護工程分局 復興工務段,2018-05-12 07:20:00,2019-07-23 08:40:00,省道台7線28K+600~28K+600,道路 道路落石,零星落石,6.293600e+05,437.1
1235,交通部公路局 北區養護工程分局 復興工務段,2018-05-12 00:00:00,2019-07-23 00:00:00,省道台7線28K+600~28K+600,道路 道路落石,零星落石,6.292800e+05,437.0
3165,交通部公路局 東區養護工程分局 太魯閣工務段,2020-01-27 08:30:00,2020-02-27 08:47:00,省道台8線165K+000~165K+000,道路 道路落石,零星落石，車道可通行，不影響交通。,4.465700e+04,31.0
4938,交通部公路局 東區養護工程分局 太魯閣工務段,2021-10-10 14:51:00,2021-11-10 14:51:00,省道台8線158K+400~158K+400,道路 道路落石,零星落石，車道可通行，不影響交通。,4.464000e+04,31.0


### 9.3.1 道路落石資料人工處理

透過管制原因判斷及日期判斷，通常很明顯都是剛好差一個月或是一年，因為日完全相同，然後管制原因又是零星落石

* idx 292 : 實際解除時間錯誤(年份) 2017-07-01 07:00:00
* idx 735 : 實際解除時間錯誤(月份) 2018-08-29 15:30:00
* idx 1234 : 移除，單一路段、零星落石，但修復時間應該是全填錯了
* idx 1235 : 移除，單一路段、零星落石，但修復時間應該是全填錯了
* idx 3165 : 實際解除時間錯誤(月份) 2020-01-27 08:47:00
* idx 4938 : 移除，不影響交通，且日期調整後為 0 
* idx 5179 : 實際解除時間錯誤(年份) 2020-04-19 10:10:00
* idx 5454 : 實際解除時間錯誤(年份) 2022-01-07 00:00:00
* idx 8208 : 實際解除時間錯誤(年份、月份) 2024-11-20 12:00:00
* idx 9514 : 移除 無法判斷


In [13]:
df[(df["Duration (days)"] > 30) & (df["災害主次類別"] == "道路 道路落石")]

,轄管局處段,通報時間,實際解除時間,路線,災害主次類別,管制原因,Duration (minutes),Duration (days)
292,交通部公路局 中區養護工程分局 谷關工務段,2017-06-14 09:11:00,2018-07-01 07:00:00,省道台8線109K+000~109K+000,道路 道路落石,NaN,5.499490e+05,381.9
735,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2018-08-29 00:19:00,2018-09-29 15:30:00,省道台18線35K+400~35K+400,道路 道路落石,邊坡坍土,4.555100e+04,31.6
1234,交通部公路局 北區養護工程分局 復興工務段,2018-05-12 07:20:00,2019-07-23 08:40:00,省道台7線28K+600~28K+600,道路 道路落石,零星落石,6.293600e+05,437.1
1235,交通部公路局 北區養護工程分局 復興工務段,2018-05-12 00:00:00,2019-07-23 00:00:00,省道台7線28K+600~28K+600,道路 道路落石,零星落石,6.292800e+05,437.0
3165,交通部公路局 東區養護工程分局 太魯閣工務段,2020-01-27 08:30:00,2020-02-27 08:47:00,省道台8線165K+000~165K+000,道路 道路落石,零星落石，車道可通行，不影響交通。,4.465700e+04,31.0
4938,交通部公路局 東區養護工程分局 太魯閣工務段,2021-10-10 14:51:00,2021-11-10 14:51:00,省道台8線158K+400~158K+400,道路 道路落石,零星落石，車道可通行，不影響交通。,4.464000e+04,31.0
5179,交通部公路局 中區養護工程分局 信義工務段,2020-04-19 06:44:00,2021-04-19 10:10:00,省道台16臨29線1K+900~1K+900,道路 道路落石,零星落石，由本段機具前往處理完畢。,5.258060e+05,365.1
5454,交通部公路局 中區養護工程分局 苗栗工務段,2021-12-22 07:42:32,2025-01-07 00:00:00,苗栗縣公館鄉省道台6線25K+400~25K+400,道路 道路落石,邊坡落石,1.600817e+06,1111.7
8208,交通部公路局 南區養護工程分局 甲仙工務段,2024-11-20 08:00:00,2025-01-20 12:00:00,高雄市六龜區省道台20線66K+0~69K+200,道路 道路落石,沿路碎石,8.808000e+04,61.2
9514,交通部公路局 中區養護工程分局 埔里工務段,2025-03-11 18:28:00,2025-11-11 09:46:00,南投縣仁愛鄉省道台14線87K+200~87K+225,道路 道路落石,無,3.522780e+05,244.6


### 9.3.2 資料修改

In [14]:
# 修正實際解除時間
df.loc[292,  "實際解除時間"] = pd.Timestamp("2017-07-01 07:00:00")
df.loc[735,  "實際解除時間"] = pd.Timestamp("2018-08-29 15:30:00")
df.loc[3165, "實際解除時間"] = pd.Timestamp("2020-01-27 08:47:00")
df.loc[5179, "實際解除時間"] = pd.Timestamp("2020-04-19 10:10:00")
df.loc[5454, "實際解除時間"] = pd.Timestamp("2022-01-07 00:00:00")
df.loc[8208, "實際解除時間"] = pd.Timestamp("2024-11-20 12:00:00")

# 移除無法判斷 + 全填錯者
df = df.drop(index=[1234, 1235,4938 ,9514]).reset_index(drop=True)

# 重新計算 Duration
# 分鐘
df["Duration (minutes)"] = (
    df["實際解除時間"] - df["通報時間"]
).dt.total_seconds() / 60

# 天數
df["Duration (days)"] = (df["Duration (minutes)"] / 1440).round(1)

# Check
df[(df["Duration (days)"] > 30) & (df["災害主次類別"] == "道路 道路落石")]

,轄管局處段,通報時間,實際解除時間,路線,災害主次類別,管制原因,Duration (minutes),Duration (days)


# 10. Cause 類別對應

| Cause | 涵蓋的 `災害主次類別` |
|---|---|
| Rockfall | 道路 道路落石、交流道 道路落石 |
| Natural Disaster | 道路 邊坡坍方、道路 土石流阻斷、道路 淹水、道路 路基流失、道路 便道沖毀、橋梁 橋梁沖毀、橋梁 便橋沖毀、橋梁 橋墩或橋面變位、隧道 淹水、交流道 淹水 |
| Other | 道路 / 橋梁 / 隧道 / 交流道 / 其他 其他、工區及周邊區域損害、設施損毀、路側設施損毀 、道路 / 橋梁 / 隧道 / 交流道 交通事故（含危險品洩漏疑慮、火災）[由於交通事故數量僅 200 筆，故不獨立為其中一筆分類]|

In [15]:
cause_map = {
    "道路 道路落石"                          : "Rockfall",
    "交流道 道路落石"                         : "Rockfall",
    "道路 邊坡坍方"                          : "Natural Disaster",
    "道路 土石流阻斷"                         : "Natural Disaster",
    "道路 淹水"                             : "Natural Disaster",
    "道路 路基流失"                          : "Natural Disaster",
    "道路 便道沖毀"                          : "Natural Disaster",
    "橋梁 橋梁沖毀"                          : "Natural Disaster",
    "橋梁 便橋沖毀"                          : "Natural Disaster",
    "橋梁 橋墩或橋面變位"                      : "Natural Disaster",
    "隧道 淹水"                             : "Natural Disaster",
    "交流道 淹水"                            : "Natural Disaster",
    "道路 交通事故(含危險品洩漏疑慮、火災)"          : "Other",
    "橋梁 交通事故(含危險品洩漏疑慮、火災)"          : "Other",
    "隧道 交通事故(含危險品洩漏疑慮、火災)"          : "Other",
    "交流道 交通事故(含危險品洩漏疑慮、火災)"         : "Other",
    "道路 其他"                             : "Other",
    "橋梁 其他"                             : "Other",
    "隧道 其他"                             : "Other",
    "交流道 其他"                            : "Other",
    "其他 其他"                             : "Other",
    "道路 工區及周邊區域損害"                    : "Other",
    "其他 工區及周邊區域損害"                    : "Other",
    "隧道 工區及周邊區域損害"                    : "Other",
    "交流道 設施損毀"                         : "Other",
    "其他 路側設施損毀"                        : "Other",
}

df["Cause"] = df["災害主次類別"].map(cause_map)

# 確認是否有未對應的類別
unmapped = df[df["Cause"].isna()]["災害主次類別"].unique()
print(f"未對應類別：{unmapped}\n")

# 類別統計
print(df["Cause"].value_counts())

# Check
df.head()

未對應類別：<StringArray>
[]
Length: 0, dtype: str

Cause
Rockfall            7790
Other               1280
Natural Disaster    1219
Name: count, dtype: int64


,轄管局處段,通報時間,實際解除時間,路線,災害主次類別,管制原因,Duration (minutes),Duration (days),Cause
0,交通部公路局 北區養護工程分局 復興工務段,2014-06-05 06:52:00,2014-06-05 09:00:00,省道台7線37K+700~37K+700,道路 道路落石,零星落石,128.0,0.1,Rockfall
1,交通部公路局 北區養護工程分局 復興工務段,2014-06-06 20:40:00,2014-06-06 22:00:00,省道台7線35K+500~35K+500,道路 道路落石,零星落石,80.0,0.1,Rockfall
2,交通部公路局 北區養護工程分局 復興工務段,2014-06-23 07:14:00,2014-06-23 09:55:00,省道台7線19K+500~19K+500,道路 道路落石,零星落石,161.0,0.1,Rockfall
3,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2014-06-01 01:00:00,2014-06-01 03:00:00,省道台18線72K+500~72K+500,道路 道路落石,零星落石,120.0,0.1,Rockfall
4,交通部公路局 雲嘉南區養護工程分局 阿里山工務段,2014-06-09 07:20:00,2014-06-09 09:20:00,省道台18線72K+500~72K+500,道路 道路落石,零星落石,120.0,0.1,Rockfall


# 11. 新增年份欄位

In [16]:
# 新增 year
df["year"] = df["通報時間"].dt.year

# 確認
print(df["year"].value_counts().sort_index())

year
2014      60
2015      88
2016     118
2017      85
2018     925
2019    1606
2020    1323
2021    1312
2022     812
2023    1567
2024    1427
2025     827
2026     139
Name: count, dtype: int64


# 12. 存檔

In [17]:
import os

OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df.to_csv(os.path.join(OUTPUT_DIR, "road_clean.csv"), index=False)
